# Lekce 08 - Vzor návrhu vícero agentů


## Nastavení


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Proč multiagentní systémy?

Úkoly v reálném světě, jako je plánování cesty, zahrnují mnoho různých druhů odbornosti — logistiku, místní znalosti, rozpočtování a další. Jeden agent, který se snaží zvládnout všechno, se rychle stává nepraktickým.

Multiagentní systémy to řeší prostřednictvím **specializace**: každý agent se zaměřuje na jednu oblast odbornosti, čímž produkuje kvalitnější výsledky než generalista. Také zlepšují **škálovatelnost** — můžete přidat nové agenty (například specialistu na lety, kritika restaurací) bez přepisování stávajícího pracovního postupu. Agentům jsou skládáni do strukturovaného potrubí, předávajíce kontext jeden druhému.


## Vytváření specializovaných agentů


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Vytváření sekvenčního pracovního postupu

`WorkflowBuilder` vám umožní propojit agenty do orientovaného grafu. Zde vytvoříme jednoduchý dvoustupňový proces: **TravelPlanner** sestaví itinerář, poté ho **TravelConcierge** zkontroluje a vylepší.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Přidání dalších agentů do pracovního postupu

Jednou z největších výhod vzoru více agentů je, jak snadné je ho rozšířit. Níže přidáváme agenta **BudgetReviewer**, který kontroluje plán vůči rozpočtu cestovatele, označuje položky, které by mohly překročit limit nákladů, a navrhuje úsporné alternativy. Pracovní postup nyní spouští tři agenty v sekvenci:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Shrnutí

V této lekci jste se naučili, jak:

1. **Vytvořit specializované agenty** — každý s konkrétní rolí (plánování, concierge, kontrola rozpočtu).
2. **Zapojit agenty do sekvenčního pracovního postupu** pomocí `WorkflowBuilder` a `add_edge`.
3. **Streamovat výstup** z pipeline s více agenty a sledovat, který agent právě mluví.
4. **Rozšířit pracovní postup** přidáním nových agentů do řetězce bez úprav stávajících.

Vzor multi-agentního návrhu udržuje každého agenta jednoduchého a zároveň produkuje bohatší, důkladněji přezkoumané výsledky, než jakého by dosáhl jediný agent.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o omezení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o co největší přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Originální dokument v jeho mateřském jazyce by měl být považován za autoritativní zdroj. Pro kritické informace se doporučuje profesionální lidský překlad. Nejsme odpovědní za jakékoli nedorozumění nebo nesprávné interpretace vzniklé použitím tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
